In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [4]:
df = pd.read_csv('pca_ex.csv')
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [5]:
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [6]:
from scipy import stats
df_numeric = df.select_dtypes(include=[np.number]) 
df_clean = df[(np.abs(stats.zscore(df_numeric))<3).all(axis = 1)]
df_clean

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [7]:
np.unique(df_clean[['Sex','ChestPainType','RestingECG','ExerciseAngina',"ST_Slope"]])

array(['ASY', 'ATA', 'Down', 'F', 'Flat', 'LVH', 'M', 'N', 'NAP',
       'Normal', 'ST', 'TA', 'Up', 'Y'], dtype=object)

In [8]:
col_to_numeric = ['Sex','ChestPainType','RestingECG','ExerciseAngina',"ST_Slope"]

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean[col_to_numeric] = df_clean[col_to_numeric].apply(le.fit_transform)
df_clean

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,1,1,140,289,0,1,172,0,0.0,2,0
1,49,0,2,160,180,0,1,156,0,1.0,1,1
2,37,1,1,130,283,0,2,98,0,0.0,2,0
3,48,0,0,138,214,0,1,108,1,1.5,1,1
4,54,1,2,150,195,0,1,122,0,0.0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,3,110,264,0,1,132,0,1.2,1,1
914,68,1,0,144,193,1,1,141,0,3.4,1,1
915,57,1,0,130,131,0,1,115,1,1.2,1,1
916,57,0,1,130,236,0,0,174,0,0.0,1,1


In [10]:
x = df_clean.drop('HeartDisease',axis='columns')
y = df_clean['HeartDisease']

In [11]:
x


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,1,1,140,289,0,1,172,0,0.0,2
1,49,0,2,160,180,0,1,156,0,1.0,1
2,37,1,1,130,283,0,2,98,0,0.0,2
3,48,0,0,138,214,0,1,108,1,1.5,1
4,54,1,2,150,195,0,1,122,0,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,3,110,264,0,1,132,0,1.2,1
914,68,1,0,144,193,1,1,141,0,3.4,1
915,57,1,0,130,131,0,1,115,1,1.2,1
916,57,0,1,130,236,0,0,174,0,0.0,1


In [12]:
y

0      0
1      1
2      0
3      1
4      0
      ..
913    1
914    1
915    1
916    1
917    0
Name: HeartDisease, Length: 899, dtype: int64

In [13]:
df_clean.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled

array([[-1.42815446,  0.515943  ,  0.2245723 , ..., -0.8229452 ,
        -0.85546862,  1.04249607],
       [-0.47585532, -1.93819859,  1.27063705, ..., -0.8229452 ,
         0.13751561, -0.62216462],
       [-1.7455875 ,  0.515943  ,  0.2245723 , ..., -0.8229452 ,
        -0.85546862,  1.04249607],
       ...,
       [ 0.3706328 ,  0.515943  , -0.82149245, ...,  1.21514774,
         0.33611246, -0.62216462],
       [ 0.3706328 , -1.93819859,  0.2245723 , ..., -0.8229452 ,
        -0.85546862, -0.62216462],
       [-1.63977649,  0.515943  ,  1.27063705, ..., -0.8229452 ,
        -0.85546862,  1.04249607]], shape=(899, 11))

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x_scaled,y,test_size=0.2,random_state=30)

In [16]:
log_reg = LogisticRegression()
log_reg.fit(X_train,y_train)
log_reg.score(X_test,y_test)

0.8388888888888889

In [17]:
model_params = {
    'svm3' : {
        'model' : SVC(gamma='auto'),
        'params' : {
            'C' : [1,10,20,30],
            'kernel':['linear','rbf']
    }},
    'logistic_reg' : {
        'model' : LogisticRegression(),
        'params' : {
            'C' : [10,1,20,30]
        }
    },
    'Decision_tree' : {
        'model': DecisionTreeClassifier(),
        'params' : {
            'max_depth' : [5,10,20,30],
            'criterion' : ['gini','entropy']
        }
    },
    'Random_forest':{
        'model' : RandomForestClassifier(),
        'params' : {
            'max_depth' : [5,10,20,30]
        }
    }
}

In [18]:
scores3 = []

for model_name,mp in model_params.items():
    clf = GridSearchCV(mp['model'],mp['params'],cv=5,return_train_score=False)
    clf.fit(X_train,y_train)
    scores3.append({
        'model' : model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

df4 = pd.DataFrame(scores3 , columns=['model' , 'best_score', 'best_params'])
df4

,model,best_score,best_params
0,svm3,0.873417,"{'C': 1, 'kernel': 'rbf'}"
1,logistic_reg,0.851166,{'C': 10}
2,Decision_tree,0.848397,"{'criterion': 'gini', 'max_depth': 5}"
3,Random_forest,0.869279,{'max_depth': 10}


In [19]:
x


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,1,1,140,289,0,1,172,0,0.0,2
1,49,0,2,160,180,0,1,156,0,1.0,1
2,37,1,1,130,283,0,2,98,0,0.0,2
3,48,0,0,138,214,0,1,108,1,1.5,1
4,54,1,2,150,195,0,1,122,0,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,3,110,264,0,1,132,0,1.2,1
914,68,1,0,144,193,1,1,141,0,3.4,1
915,57,1,0,130,131,0,1,115,1,1.2,1
916,57,0,1,130,236,0,0,174,0,0.0,1


In [20]:
x.shape

(899, 11)

In [21]:
from sklearn.decomposition import PCA

pca = PCA(0.95)
X_pca = pca.fit_transform(x)
X_pca.shape

(899, 2)

In [22]:
pca.explained_variance_ratio_

array([0.92109746, 0.05064983])

In [23]:
pca.n_components_

np.int64(2)

In [24]:
X_pca

array([[ 93.12926348,  29.67413245],
       [-16.33750689,  14.81536427],
       [ 82.66842478, -38.91589868],
       ...,
       [-68.22644416, -17.7012641 ],
       [ 40.02690223,  33.47134474],
       [-20.61151816,  37.62451392]], shape=(899, 2))

In [25]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca,y,test_size=0.2,random_state=30)


In [26]:
log_reg2 = LogisticRegression(C=10)
log_reg2.fit(X_train_pca,y_train)
log_reg2.score(X_test_pca,y_test)


0.6666666666666666

In [27]:
pca2 = PCA(n_components=5)
X_pca2 = pca2.fit_transform(x)
X_pca2.shape

(899, 5)

In [28]:
pca2.explained_variance_ratio_

array([9.21097460e-01, 5.06498292e-02, 2.25805052e-02, 5.43577180e-03,
       9.17476084e-05])

In [29]:
pca2.n_components

5

In [30]:
X_pca2

array([[ 93.12926348,  29.67413245,  10.95031818,  -9.15962775,
         -0.4187242 ],
       [-16.33750689,  14.81536427,  31.09135696,  -6.1896339 ,
         -0.29575433],
       [ 82.66842478, -38.91589868, -14.78369731, -21.15840251,
         -1.24737132],
       ...,
       [-68.22644416, -17.7012641 ,  -4.33467509,   0.3949739 ,
          0.5908128 ],
       [ 40.02690223,  33.47134474,   5.35465207,   9.19154029,
         -0.42523827],
       [-20.61151816,  37.62451392,  12.15953648, -10.98027526,
         -0.79976719]], shape=(899, 5))

In [31]:
X_train_pca2, X_test_pca2, y_train, y_test = train_test_split(X_pca2,y,test_size=0.2,random_state=30)


In [32]:
log_reg3 = LogisticRegression(C=10)
log_reg3.fit(X_train_pca2,y_train)
log_reg3.score(X_test_pca2,y_test)

0.8111111111111111